# Refusal-direction baseline on Colab Pro

This notebook is the Colab Pro entry point for GPU-dependent baseline runs. It assumes the runtime has a CUDA GPU enabled.

Before running the notebook, choose **Runtime → Change runtime type → Hardware accelerator → GPU**.

## Verify GPU runtime

If this cell fails, switch the runtime to a GPU runtime before continuing.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "CUDA is unavailable. In Colab, choose Runtime → Change runtime type → GPU."
)
print(torch.cuda.get_device_name(0))

## Get the project code

If you opened this notebook from the repository in Colab, the checkout may already exist. Otherwise, set `REPO_URL` to your GitHub repository URL and run the clone cell.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/cayblood/refusal_direction_extended.git"
PROJECT_DIR = Path("/content/refusal_direction_extended")

if PROJECT_DIR.exists():
    print(f"Using existing checkout: {PROJECT_DIR}")
else:
    if not REPO_URL:
        raise ValueError(
            "Set REPO_URL to your repository URL, then rerun this cell."
        )
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}

## Install project tooling and dependencies

Colab runtimes are ephemeral, so this installs `uv` and syncs the project dependencies in the runtime.

In [ ]:
!pip install -q uv
!uv sync

## Optional: Hugging Face token

Qwen models are public, but setting `HF_TOKEN` avoids anonymous rate limits. This cell prompts securely and stores the token only in the current Colab runtime environment. Do not paste tokens directly into saved notebook cells.

You can skip this cell if downloads are already fast enough.

In [ ]:
import os
import subprocess
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")

# Log the Hugging Face CLI into this runtime as well, so downloads and
# libraries that read the local HF token cache can use the token.
subprocess.run(
    ["uv", "run", "hf", "auth", "login", "--token", os.environ["HF_TOKEN"]],
    check=True,
)
subprocess.run(["uv", "run", "hf", "auth", "whoami"], check=True)

## Verify project environment

This uses `uv run` so it checks the project environment, not only the Colab notebook kernel.


In [ ]:
import subprocess
import textwrap

code = textwrap.dedent(
    """
    from importlib.metadata import version

    import torch

    print("torch", version("torch"))
    print("transformer-lens", version("transformer-lens"))
    print("transformers", version("transformers"))
    print("datasets", version("datasets"))
    print("cuda", torch.cuda.is_available())
    print("gpu", torch.cuda.get_device_name(0))
    """
)

subprocess.run(["uv", "run", "python", "-c", code], check=True)

## Download/cache models

This downloads both Qwen2.5 Instruct models into the active Colab runtime cache. If you want persistent caching, configure Hugging Face cache paths to mounted Drive before running this cell.

In [ ]:
!uv run hf download Qwen/Qwen2.5-1.5B-Instruct --repo-type model
!uv run hf download Qwen/Qwen2.5-3B-Instruct --repo-type model

## Run baseline generations

The script requires CUDA by default, matching the project convention for GPU-dependent tasks.

In [ ]:
!uv run python scripts/baseline.py --device cuda

## Useful variants


In [ ]:
# Run only the smaller model.
# MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
# !uv run python scripts/baseline.py --device cuda --model {MODEL}

# Generate longer completions.
# !uv run python scripts/baseline.py --device cuda --max-new-tokens 256